# Kaggle project: Titanic

This notebook contains first version of ML prediction of Titanic survivors. I plan to do other versions with more sophiscitated methods.

In [17]:
%%capture
%pip install -q kaggle kagglehub

In [18]:
import json
import subprocess
from pathlib import Path

import kagglehub
import pandas as pd
from IPython.display import HTML, display
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import numpy as np

## 1. Data description and loading

In [19]:
result = subprocess.run(
    [
        "kaggle", "competitions", "pages", "list",
        "-c", "titanic",
        "--page-name", "data-description",
        "--content",
        "--format", "json",
        "-q",
    ],
    capture_output=True,
    text=True,
    check=True,
)

pages = json.loads(result.stdout)
display(HTML(pages[0]["content"]))

Variable,Definition,Key
survival,Survival,"0 = No, 1 = Yes"
pclass,Ticket class,"1 = 1st, 2 = 2nd, 3 = 3rd"
sex,Sex,
Age,Age in years,
sibsp,# of siblings / spouses aboard the Titanic,
parch,# of parents / children aboard the Titanic,
ticket,Ticket number,
fare,Passenger fare,
cabin,Cabin number,
embarked,Port of Embarkation,"C = Cherbourg, Q = Queenstown, S = Southampton"


In [20]:
data_path = Path(kagglehub.competition_download("titanic"))

train = pd.read_csv(data_path / "train.csv")
test = pd.read_csv(data_path / "test.csv")

In [21]:
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
display(train.head())

Train shape: (891, 12)
Test shape:  (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Data quality

In [22]:
print("TRAIN")
train.info()

print("\nTEST")
test.info()

TRAIN
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

TEST
<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-n

In [23]:
def missing_summary(dataframe):
    return (
        pd.DataFrame({
            "Count": dataframe.isna().sum(),
            "Percent": (dataframe.isna().mean() * 100).round(1),
        })
        .sort_values("Percent", ascending=False)
    )

print("TRAIN — missing values")
display(missing_summary(train))

print("TEST — missing values")
display(missing_summary(test))

TRAIN — missing values


,Count,Percent
Cabin,687,77.1
Age,177,19.9
Embarked,2,0.2
PassengerId,0,0.0
Name,0,0.0
Pclass,0,0.0
Survived,0,0.0
Sex,0,0.0
Parch,0,0.0
SibSp,0,0.0


TEST — missing values


,Count,Percent
Cabin,327,78.2
Age,86,20.6
Fare,1,0.2
Name,0,0.0
Pclass,0,0.0
PassengerId,0,0.0
Sex,0,0.0
Parch,0,0.0
SibSp,0,0.0
Ticket,0,0.0


In [24]:
target_distribution = (
    train["Survived"]
    .value_counts()
    .sort_index()
    .rename_axis("Survived")
    .to_frame("Count")
)
target_distribution["Percent"] = (
    target_distribution["Count"] / len(train) * 100
).round(1)

display(target_distribution)

,Count,Percent
Survived,,
0,549,61.6
1,342,38.4


In [25]:
print("Age summary")
display(train[["Age"]].describe().T)
display(train["Age"].value_counts(dropna=False).rename("Count"))

train["Age"] = np.ceil(train["Age"])
test["Age"] = np.ceil(test["Age"])

print("Embarked counts")
display(train["Embarked"].value_counts(dropna=False).rename("Count"))

print("Ten most common Cabin values")
display(train["Cabin"].value_counts(dropna=False).head(10).rename("Count"))

print("Fare value counts")
display(test["Fare"].value_counts(dropna=False).rename("Counts"))
train["Fare"] = np.ceil(train["Fare"])
test["Fare"] = np.ceil(test["Fare"])

Age summary


,count,mean,std,min,25%,50%,75%,max
Age,714.0,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0


Age
NaN      177
24.00     30
22.00     27
18.00     26
28.00     25
        ... 
24.50      1
0.67       1
0.42       1
34.50      1
74.00      1
Name: Count, Length: 89, dtype: int64

Embarked counts


Embarked
S      644
C      168
Q       77
NaN      2
Name: Count, dtype: int64

Ten most common Cabin values


Cabin
NaN            687
G6               4
C23 C25 C27      4
B96 B98          4
F33              3
E101             3
F2               3
D                3
C22 C26          3
C123             2
Name: Count, dtype: int64

Fare value counts


Fare
7.7500      21
26.0000     19
8.0500      17
13.0000     17
7.8958      11
            ..
13.8625      1
7.7208       1
90.0000      1
108.9000     1
22.3583      1
Name: Counts, Length: 170, dtype: int64

## 3. Exploratory data analysis

In [26]:
cabin_known_analysis = (
    train.assign(CabinKnown=train["Cabin"].notna())
    .groupby("CabinKnown")["Survived"]
    .agg(["count", "mean"])
)
cabin_known_analysis["mean"] = (cabin_known_analysis["mean"] * 100).round(1)
cabin_known_analysis = cabin_known_analysis.rename(
    columns={"count": "Passengers", "mean": "Survival rate (%)"}
)

display(cabin_known_analysis)

,Passengers,Survival rate (%)
CabinKnown,,
False,687,30.0
True,204,66.7


In [27]:
for column in ["Sex", "Pclass", "Embarked", "SibSp", "Parch"]:
    summary = (
        train.groupby(column, dropna=False)["Survived"]
        .agg(["count", "mean"])
        .rename(columns={"count": "Passengers", "mean": "Survival rate"})
    )
    summary["Survival rate"] = (summary["Survival rate"] * 100).round(1)

    print(column)
    display(summary.sort_values("Survival rate", ascending=False))

Sex


,Passengers,Survival rate
Sex,,
female,314,74.2
male,577,18.9


Pclass


,Passengers,Survival rate
Pclass,,
1,216,63.0
2,184,47.3
3,491,24.2


Embarked


,Passengers,Survival rate
Embarked,,
NaN,2,100.0
C,168,55.4
Q,77,39.0
S,644,33.7


SibSp


,Passengers,Survival rate
SibSp,,
1,209,53.6
2,28,46.4
0,608,34.5
3,16,25.0
4,18,16.7
5,5,0.0
8,7,0.0


Parch


,Passengers,Survival rate
Parch,,
3,5,60.0
1,118,55.1
2,80,50.0
0,678,34.4
5,5,20.0
4,4,0.0
6,1,0.0


In [28]:
print(f"Duplicated rows: {train.duplicated().sum()}")
display(train.nunique().sort_values().rename("Unique values").to_frame())

Duplicated rows: 0


,Unique values
Survived,2
Sex,2
Pclass,3
Embarked,3
Parch,7
SibSp,7
Age,70
Fare,89
Cabin,147
Ticket,681


## 4. Data preparation

Based on the data quality analysis:

- Preserve sector of cabins as `CabinSector`, then remove the original high-cardinality `Cabin` feature.
- Impute missing `Age` and `Fare` values with medians learned only from the training split.
- Impute missing `Embarked` values with the most frequent value learned only from the training split.

In [33]:
X = train.drop(columns=["Survived"]).copy()
y = train["Survived"].copy()
test_features = test.copy()

X["CabinSector"] = X["Cabin"].str[0]
test_features["CabinSector"] = test_features["Cabin"].str[0]

X = X.drop(columns=["Cabin", "Ticket"])
test_features = test_features.drop(columns=["Cabin", "Ticket"])

In [34]:
stratify_key = X["Sex"].astype(str) + "_" + y.astype(str)

X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=stratify_key,
)

split_distribution = pd.DataFrame({
    "All data (%)": stratify_key.value_counts(normalize=True) * 100,
    "Train (%)": stratify_key.loc[X_train.index].value_counts(normalize=True) * 100,
    "Validation (%)": stratify_key.loc[X_validation.index].value_counts(normalize=True) * 100,
}).round(1)

print(f"Training rows:   {len(X_train)}")
print(f"Validation rows: {len(X_validation)}")
display(split_distribution)

Training rows:   712
Validation rows: 179


,All data (%),Train (%),Validation (%)
male_0,52.5,52.5,52.5
female_1,26.2,26.1,26.3
male_1,12.2,12.2,12.3
female_0,9.1,9.1,8.9


In [35]:
X_train = X_train.copy()
X_validation = X_validation.copy()

numeric_columns = ["Age", "Fare"]
categorical_columns = ["Embarked"]

numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

X_train[numeric_columns] = numeric_imputer.fit_transform(
    X_train[numeric_columns]
)
X_validation[numeric_columns] = numeric_imputer.transform(
    X_validation[numeric_columns]
)

X_train[categorical_columns] = categorical_imputer.fit_transform(
    X_train[categorical_columns]
)
X_validation[categorical_columns] = categorical_imputer.transform(
    X_validation[categorical_columns]
)

imputation_values = pd.DataFrame(
    {
        "Imputation value": [*numeric_imputer.statistics_, *categorical_imputer.statistics_]
    },
    index=[*numeric_columns, *categorical_columns]
)
display(imputation_values)

,Imputation value
Age,28.0
Fare,14.0
Embarked,S
